<a href="https://colab.research.google.com/github/somendrew/LangGraph_tutorial/blob/main/LangGraph_Concept.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LangGraph Concepts: A Deep Dive

In [ ]:
# Install necessary packages
!pip install -qqq langchain-openai langgraph

### Why LangGraph

LangChain made it easy to string prompts together: prompt | llm | parser. Clean, readable, good for simple tasks.

But real AI agents rarely go in a straight line. A customer support agent reads a message, decides whether to search a knowledge base or call a tool, retries if something fails, and needs to remember the full conversation. A linear chain cannot do that.

LangGraph handles branching, looping, retries, persistence, and human-in-the-loop checkpoints through explicit primitives that make agent behavior visible at every step.

LangChain's 2026 State of Agent Engineering report found that over 70% of production agents adopted a graph structure rather than a simple linear chain. Real business processes rarely go straight to the end.

The six concepts below are how that graph structure actually works.

## 1. Concept: State

State is the shared notebook that every step in your workflow can read from and write to.

Before LangGraph, agent state was scattered: some in variables, some in memory, some in the conversation history. You could never be sure what any given step actually knew. LangGraph fixes this by making state explicit and typed upfront.

Every node in the graph can see and change any field in the state. Start with only the fields you actually need. Most guides skip this part: don't design a 20-field state upfront. Let the requirements emerge.

The `Annotated[list, add]` is worth understanding. By default, when two nodes update the same field, the second one overwrites the first. Adding `add` as an annotation tells LangGraph to merge lists instead of replacing them. Use it for messages and accumulated results. Use plain types for current status fields like which step you are on.

**Common beginner mistake:** Storing entire LLM responses with usage metadata in state. One team building a document processing agent stored raw LLM responses in state. At 50 documents, the state object was 180KB per checkpoint. Postgres writes climbed to 400ms and started affecting response time. The fix was stripping state to just what downstream nodes actually need.

In [1]:
from typing import TypedDict, Annotated
from operator import add

class AgentState(TypedDict):
    question: str           # the user's input
    answer: str             # what the agent produces
    messages: Annotated[list, add]  # conversation history, grows over time

## 2. Concept: Nodes

A node is just a Python function. It receives the current state, does some work, and returns the fields it wants to update.

That's it. If you can write a function in Python, you can build a node. Call an LLM, hit a database, change some text, fire off an API call. Whatever Python can do. There is just one rule: take state in, send state updates back.

**Common beginner mistake:** Returning the full state from a node. You only need to return the fields you changed. Returning everything causes subtle overwrite bugs when multiple nodes update overlapping fields.

LangGraph nodes are just Python functions. The framework is simpler than it looks.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.schema import HumanMessage

llm = ChatOpenAI(model="gpt-4o-mini")

def answer_node(state: AgentState) -> dict:
    # Reads from state, returns only the fields that changed
    response = llm.invoke([HumanMessage(content=state["question"])])
    return {"answer": response.content}

def refine_node(state: AgentState) -> dict:
    prompt = f"Make this clearer: {state['answer']}"
    response = llm.invoke([HumanMessage(content=prompt)])
    return {"answer": response.content}

## 3. Concept: Edges

Edges are the wires connecting nodes. They tell LangGraph which node to run next.

There are two kinds. Direct edges always go the same way: when node A finishes, always run node B. Conditional edges choose where to go based on the current state: when node A finishes, check the state and decide.

**Common beginner mistake:** Forgetting END. If you do not connect the last node to END, the graph runs forever waiting for a next step that never arrives. This is the most common cause of infinite loops in early LangGraph code.

In [ ]:
from langgraph.graph import StateGraph, END

graph = StateGraph(AgentState)
# Add nodes
graph.add_node("answer", answer_node)
graph.add_node("refine", refine_node)
# Direct edge: answer always goes to refine
graph.add_edge("answer", "refine")
# Direct edge: refine goes to END
graph.add_edge("refine", END)
graph.set_entry_point("answer")

# Compile the graph (this is a conceptual example, actual compilation may require more setup)
app = graph.compile()

## 4. Concept: Conditional Edges

This is where LangGraph becomes genuinely powerful. Instead of always going to the same next node, a conditional edge inspects the state and returns the name of the node to run next.

Think of it like a railway switch. The train is the state. The switch checks the state and sends the train down one of two tracks.

Conditional edges are the agent's decision mechanism. A function inspects state and returns the next node name. This is how "should I use another tool or stop?" is implemented.

**Common beginner mistake:** Returning a node name that does not exist in the graph. The error message is cryptic and finding the typo takes longer than it should. Always match return values exactly to the keys in your edge mapping dictionary.

In [4]:
def route_based_on_quality(state: AgentState) -> str:
    # Check the current answer quality
    if len(state["answer"]) < 50:
        return "refine"   # too short, needs more work
    return "done"         # good enough, finish

# Example of adding a conditional edge (requires 'graph' to be defined and compiled)
graph.add_conditional_edges(
    "answer",           # from this node
    route_based_on_quality,  # use this function to decide
    {
        "refine": "refine",  # if function returns "refine", go to refine node
        "done": END          # if function returns "done", end the graph
    }
)

## 5. Concept: Checkpointing

Checkpointing is how LangGraph gives your agent persistent memory.

Without a checkpointer, every call to `app.invoke()` starts fresh. The agent has no memory of past sessions. Add a checkpointer and the agent saves its state after every node transition, keyed by a thread ID. The next call with the same thread ID picks up exactly where it left off.

The fix to a production crash that would have taken a week of custom serialization, a Redis state cache, and a session reconstruction function took 45 minutes with LangGraph checkpointing.

Use `MemorySaver` in development. Use `SqliteSaver` for single-server production. Use `PostgresSaver` when you need multiple servers to share the same state.

**Common beginner mistake:** Using `MemorySaver` in production. It stores everything in RAM. Restart the server and all agent state is gone.

In [5]:
from langgraph.checkpoint.memory import MemorySaver  # dev only
from langgraph.checkpoint.sqlite import SqliteSaver  # single-server prod
from langgraph.checkpoint.postgres import PostgresSaver  # multi-instance prod

checkpointer = MemorySaver()
app = graph.compile(checkpointer=checkpointer)
# thread_id groups all interactions for one "session"
config = {"configurable": {"thread_id": "user-session-42"}}
# First call: agent runs and saves state
app.invoke({"question": "What is LangGraph?"}, config)
# Second call with same thread_id: picks up where it left off
app.invoke({"question": "Show me a code example"}, config)

## 6. Concept: Human-in-the-Loop (Interrupts)

60% of production agent systems added human intervention points. Not fully autonomous agents, but ones that pause at key decision points, wait for human confirmation, and then continue.

LangGraph implements this through `interrupt_before`. You specify which node should trigger a pause. The graph stops before entering that node, waits for a human to review and optionally update the state, then resumes.

**Common beginner mistake:** Trying to implement human approval with a conversation turn instead of an interrupt. Asking the model "should I proceed?" and trusting its answer is not a human-in-the-loop. It is asking the agent to approve its own actions.

An agent that approves its own risky decisions is not supervised. It is theatrical.

In [6]:
Compile with interrupt_before to pause before the risky node
app = graph.compile(
    checkpointer=checkpointer,
    interrupt_before=["send_email"]  # pause before this node
)
config = {"configurable": {"thread_id": "task-99"}}
# Graph runs until it hits send_email, then pauses
app.invoke({"task": "Draft and send a refund email"}, config)
# A human reviews the draft here, optionally updates state
graph.update_state(config, {"draft": "Updated email text"})
# Resume from where it paused, with human-reviewed state
app.invoke(None, config)